# Phase 7 — Generation / answer synthesis

**LLM choice: Cohere `command-a-03-2025`** (current flagship Command model), called via `cohere.ClientV2.chat` with the native `documents` + `citation_options` parameters instead of manual citation prompting. Reasoning:
- Same vendor/key already configured for embeddings (`COHERE_API_KEY`) -- no new key setup.
- Cohere's chat API is purpose-built for RAG: pass retrieved chunks as `documents`, get back structured citations that map spans of the answer to specific document ids -- this satisfies "cite law + article number" *structurally*, not just by hoping the model follows a prompt instruction.
- Command models are natively multilingual (AR/EN), matching the project's bilingual requirement.

Implementation is `src/generation.py` (`Generator` class, importable for the future FastAPI backend), same pattern as `src/retrieval.py`. The system prompt enforces: answer only from provided documents, cite law + article, admit honestly when the documents don't have the answer (no hallucination), match the question's language, and close with a "not legal advice" disclaimer.

This notebook wires `Retriever.retrieve()` -> `Generator.generate()` into one pipeline and spot-checks it on a handful of representative cases (AR, EN, and an out-of-domain question to test honest refusal). The formal 40-question accuracy run is Phase 8, not here.

Kernel: `rag311`.

In [1]:
import sys
from pathlib import Path

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.retrieval import Retriever
from src.generation import Generator

retriever = Retriever()
generator = Generator()

def ask(query, top_k=5):
    chunks = retriever.retrieve(query, top_k=top_k)
    result = generator.generate(query, chunks)
    print(f"Q: {query}\n")
    print(f"A: {result['answer']}\n")
    print("citations:")
    for c in result["citations"]:
        print(f"  - {c['law']} art.{c['article_number']} ({c['language']}): \"{c['cited_text']}\"")
    return result

## In-scope questions (Arabic and English)

In [2]:
_ = ask("كم يوم إجازة سنوية يستحقها الموظف؟")

Q: كم يوم إجازة سنوية يستحقها الموظف؟

A: يحق للموظف الحصول على إجازة سنوية مدفوعة الأجر لمدة أربعة عشر يومًا عن كل سنة خدمة، وفقًا لـقانون العمل لعام 1996، المادة 61. ومع ذلك، إذا أمضى الموظف خمس سنوات متتالية في خدمة نفس صاحب العمل، تزداد مدة الإجازة السنوية إلى واحد وعشرين يومًا.

**ملاحظة**: هذه المعلومات عامة ولا تعتبر استشارة قانونية. يُنصح باستشارة محامٍ مرخص للحصول على المشورة القانونية المخصصة لحالتك.

citations:
  - Labor Law 1996 art.61 (ar): "أربعة عشر يومًا"
  - Labor Law 1996 art.61 (en): "أربعة عشر يومًا"
  - Labor Law 1996 art.61 (ar): "قانون العمل لعام 1996، المادة 61"
  - Labor Law 1996 art.61 (en): "قانون العمل لعام 1996، المادة 61"
  - Labor Law 1996 art.61 (ar): "خمس سنوات متتالية"
  - Labor Law 1996 art.61 (en): "خمس سنوات متتالية"
  - Labor Law 1996 art.61 (ar): "واحد وعشرين يومًا."
  - Labor Law 1996 art.61 (en): "واحد وعشرين يومًا."


In [3]:
_ = ask("What compensation am I entitled to if I'm terminated without notice?")

Q: What compensation am I entitled to if I'm terminated without notice?

A: If you are terminated without notice, you may be entitled to compensation depending on the circumstances of your termination.

According to Labor Law 1996, Article 25, if the termination is deemed arbitrary and in violation of the law, you may be entitled to compensation equivalent to half a month's wages for each year of service, with a minimum limit of two monthly wages, in addition to the notice allowance and other entitlements specified in Articles 32 and 33.

Furthermore, Article 32 states that a worker whose service ends for any reason is entitled to receive an end-of-service remuneration at the rate of one month's wage for each year of actual service.

However, please note that the employer may dismiss the worker without notice in certain cases, as outlined in Article 28.

**Disclaimer:** This is general information, not legal advice. Please consult a licensed lawyer for your specific situation.

citatio

## Cross-lingual case

Arabic question whose best-supporting article may come back as an English-language chunk (Phase 6's known cross-lingual weak spot) -- check the answer still comes back in Arabic and cites correctly regardless of which language the source chunk is in.

In [4]:
_ = ask("متى تنطبق عليّ أحكام إصابات العمل في قانون العمل بدلاً من قانون الضمان الاجتماعي؟")

Q: متى تنطبق عليّ أحكام إصابات العمل في قانون العمل بدلاً من قانون الضمان الاجتماعي؟

A: تنطبق أحكام إصابات العمل في قانون العمل على العمال الذين لا تسري عليهم أحكام قانون الضمان الاجتماعي المعمول به.

**ملاحظة**: هذه المعلومات عامة ولا تعتبر استشارة قانونية. يُنصح بالرجوع إلى محامٍ مرخص للحصول على المشورة القانونية الخاصة بحالتك.

citations:
  - Labor Law 1996 art.86 (ar): "على العمال الذين لا تسري عليهم أحكام قانون الضمان الاجتماعي المعمول به"


## Out-of-domain question

The retriever will still return its top-k nearest chunks (it has no "none of these are relevant" option), so this checks whether the *generation* step honestly refuses instead of forcing an answer out of irrelevant context.

In [5]:
_ = ask("What is the minimum wage in Germany?")

Q: What is the minimum wage in Germany?

A: I'm sorry, I couldn't find any information about the minimum wage in Germany. However, I did find information about the minimum wage in Jordan.

According to Labor Law 1996, Article 52, a Tripartite Committee is responsible for determining the minimum wage in Jordan, taking into account the cost of living indicators. The committee's decisions are published in the Official Gazette.

Please note that this is general information and not legal advice. For specific situations, it's recommended to consult a licensed lawyer.

citations:
  - Labor Law 1996 art.52 (en): "Labor Law 1996, Article 52"
  - Labor Law 1996 art.52 (en): "Tripartite Committee"
  - Labor Law 1996 art.52 (en): "determining the minimum wage"
  - Labor Law 1996 art.52 (en): "cost of living indicators."
  - Labor Law 1996 art.52 (en): "published in the Official Gazette."


## Results\n\nAll four spot checks came back solid:\n1. **AR annual leave**: correct (14 days base, 21 after 5 consecutive years), cites Article 61, disclaimer in Arabic.\n2. **EN termination compensation**: comprehensive -- cites Article 25 (arbitrary-dismissal compensation), Article 32 (end-of-service remuneration), Article 28 (when dismissal without notice is allowed), disclaimer included.\n3. **Cross-lingual work-injury question** (Phase 6's known weak spot): answered correctly in Arabic citing Article 86, regardless of which language the retrieved chunk happened to be in.\n4. **Out-of-domain (German minimum wage)**: no hallucination -- explicitly says it found nothing about Germany -- but then volunteers Jordan's minimum-wage rule (Article 52) unprompted instead of just stopping at \"not found.\" Not wrong, but a judgment call worth noting: the system prompt says \"say so honestly,\" not \"redirect to related information.\" Leaving as-is for now since it's harmless and arguably more useful, but worth watching in Phase 8's formal eval in case it starts papering over genuine gaps instead of surfacing them.\n\n(One fixed issue along the way: `command-a-03-2025` doesn't support `citation_options.mode=\"ACCURATE\"` -- switched to the default `\"ENABLED\"` mode, which works and returns the same structured per-span citations.)\n\nPhase 7 done. `Generator.generate(query, chunks)` in `src/generation.py` is the reusable function -- combined with `Retriever.retrieve()` this is the full retrieve-then-generate pipeline, ready for Phase 8's formal 40-question eval and Phase 9's FastAPI backend.